# Cohere and Hugging Face integration: the fixes, live

A tour of AuditKIT's Cohere and `hf:` integration. Each section shows one fix from the #15 plan (with #27's follow-ups) doing its job, and ends with a PASS/FAIL check collected in the final checklist.

| § | Fix | Shown with |
|---|---|---|
| 1 | `hf:` native tool calling; **G13** `RunConfig.chat_template_kwargs` | Qwen3-1.7B, live |
| 2 | **G14** `f1_score` ignores case, punctuation and articles | Qwen3 answers, live |
| 3 | Tiny Aya: native tools **refused with advice**, prompt mode works | Tiny Aya Global, live (gated) |
| 4 | **G11** OpenAI-style string arguments render as objects in Command R7B's template | the real Command R7B template |
| 5 | **G7–G10** Cohere tool-call formats: call ids, parameterless calls, `directly-answer`, truncated action lists | Cohere's documented output formats |
| 6 | Command R7B native tool calling | Command R7B, live (gated, 24 GB+ GPU) |
| 7 | Cohere's hosted API through `api:` (native and parallel calls, multi-turn) | Cohere API, live |
| 8 | **G1/G2** the harness runs a *text* model's tool calls, with a generation budget; **#27** no duplicated calls in the fed-back history | Qwen3, live |
| 9 | **G3–G5** AgentTune TraceLogger calls without names are unscored, not 0.0 | a TraceLogger-format file |
| 10 | **G6** an `agent:` reply without a transcript is unscored | a local agent endpoint |
| 11 | **G16** setup errors are raised at once, not retried | `hf:` with a model id that doesn't exist |
| 12 | Checklist | – |

**Runtime:** a Colab GPU. A T4 is enough for everything except §6, which needs 24 GB+.

**Secrets:**
- `GITHUB_TOKEN` (to install AuditKIT);
- `HF_TOKEN` with the Tiny Aya / Command R7B licences accepted (§3, §6);
- `CO_API_KEY` (§7).

A section whose requirement is missing is skipped with a note. It's marked SKIP in the checklist, not FAIL.

In [ ]:
# ---- settings ----
QWEN = "Qwen/Qwen3-1.7B"
TINY_AYA = "CohereLabs/tiny-aya-global"
COMMAND_R7B = "CohereLabs/c4ai-command-r7b-12-2024"
COHERE_API_MODEL = "command-r7b-12-2024"
COHERE_BASE = "https://api.cohere.ai/compatibility/v1"

In [ ]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at BRANCH (#1's branch, where #10 and every #15 fix are merged) and installs it.
# Locally: uses the auditkit already importable (e.g. the repo's .venv kernel).
import os, subprocess, sys
BRANCH = "feat/rag-agent-evals"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'HF_TOKEN', 'CO_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass                          # secret not set / not granted to this notebook
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[transformers,requests]"], check=True)
    sys.path.insert(0, "/content/AuditKIT-Internal/src")
import auditkit as ak
print("auditkit", ak.__version__, "| secrets:", {k: bool(os.environ.get(k)) for k in ('HF_TOKEN', 'CO_API_KEY')})

In [ ]:
import json, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
import pandas as pd
import torch
from auditkit import Sample
from auditkit.runspec import RunConfig
from auditkit.errors import CapabilityError
from auditkit.model.hf_gen import HFGenModel
from auditkit.trace import parse_tool_calls
pd.set_option("display.max_colwidth", 90)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
REPO = Path("/content/AuditKIT-Internal") if IN_COLAB else Path(ak.__file__).resolve().parents[2]
HF_TOKEN = os.environ.get("HF_TOKEN")
CHECKS = {}                                   # fix -> "PASS" / "FAIL" / "SKIP (...)"
def check(name, ok, skip=None):
    CHECKS[name] = f"SKIP ({skip})" if skip else ("PASS" if ok else "FAIL")
    print(f"{CHECKS[name]:>6}  {name}")
print("device", DEVICE, f"({GPU_GB:.0f} GB)" if GPU_GB else "")
print(ak.check_compat())

## 1 · `hf:` native tool calling, and `chat_template_kwargs` (G13)

`ToolCallAdapter()` hands the tool schemas to the model's **own chat template**, and AuditKIT parses the calls from the reply. Qwen3 is a *thinking* model: it reasons in `<think>…</think>` first unless its template gets `enable_thinking=False`. `RunConfig.chat_template_kwargs` passes that to every request's template, so the same tasks run with thinking on and off:
- thinking off: short answer-only replies, and the same tool calls;
- thinking on: it needs far more tokens.

In [ ]:
TOOLS = [
    {"type": "function", "function": {"name": "get_weather", "description": "Current weather for a city",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "get_time", "description": "Current local time in a city",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
]
W = lambda c: {"name": "get_weather", "arguments": {"city": c}}
TASKS = [
    Sample(id="single", input="What's the weather in Paris?", tools=TOOLS, expected_tool_calls=[[W("Paris")]]),
    Sample(id="parallel", input="What's the weather in Paris and in Rome? Check both at once.", tools=TOOLS, expected_tool_calls=[[W("Paris"), W("Rome")]]),
    Sample(id="pick_tool", input="What time is it in Tokyo?", tools=TOOLS, expected_tool_calls=[[{"name": "get_time", "arguments": {"city": "Tokyo"}}]]),
    Sample(id="no_tool", input="What is 2+2? Answer directly, without tools.", tools=TOOLS, expected_tool_calls=[]),
]
SCORERS = lambda: [ak.ToolCallF1(), ak.TrajectoryMatch(), ak.ParallelToolCalls(), ak.ToolCallValidity()]
qwen = HFGenModel(QWEN, device=DEVICE, name=f"hf:{QWEN}")
off = ak.evaluate(TASKS, model=qwen, adapter=ak.ToolCallAdapter(), scorers=SCORERS(),
                  config=RunConfig(max_tokens=256, temperature=0.0, chat_template_kwargs={"enable_thinking": False}))
on = ak.evaluate(TASKS, model=qwen, adapter=ak.ToolCallAdapter(), scorers=SCORERS(), config=RunConfig(max_tokens=1024, temperature=0.0))
mean_len = lambda r: sum(len(p.raw_output or "") for p in r.predictions) / len(r.predictions)
print("thinking off:", {k: round(v, 2) for k, v in sorted(off.headline.items()) if k in ("tool_call_f1", "parallel_detection", "trajectory_strict")},
      f"| mean reply {mean_len(off):.0f} chars")
print("thinking on :", {k: round(v, 2) for k, v in sorted(on.headline.items()) if k in ("tool_call_f1", "parallel_detection", "trajectory_strict")},
      f"| mean reply {mean_len(on):.0f} chars")
for p in off.predictions:
    print(f"  {p.sample_id:10s} -> {[(c.name, c.arguments) for c in parse_tool_calls(p.raw_output or '')]}")
check("hf: native tool calling (Qwen3)", off.headline.get("tool_call_f1", 0) >= 0.75)
check("G13 chat_template_kwargs reaches the template", mean_len(off) < mean_len(on) and off.fingerprint != on.fingerprint)

## 2 · `f1_score` normalises like SQuAD (G14)

`f1_score` compares answer tokens after lowercasing and removing punctuation and articles, so "Paris." and "paris" are the same answer. `F1Score(normalize=False)` keeps the old raw split, and that setting is part of the metric's identity.

In [ ]:
from auditkit.metrics.code import F1Score
QA = [("What is the capital of France? Answer with one word.", "paris"), ("Which planet is called the Red Planet? One word.", "mars"),
      ("What is the largest ocean? Answer in two words.", "the pacific")]
qa = [Sample(id=f"qa{i}", input=q, target=a) for i, (q, a) in enumerate(QA)]
cfg = RunConfig(max_tokens=32, temperature=0.0, chat_template_kwargs={"enable_thinking": False})
norm = ak.evaluate(qa, model=qwen, scorers=[F1Score()], config=cfg)
raw = ak.evaluate(qa, model=qwen, scorers=[F1Score(normalize=False)], config=cfg)
for p in norm.predictions:
    print(f"  gold {p.expected!r:14} answer {(p.raw_output or '').strip()!r}")
print("f1 normalised:", round(norm.headline["f1_score"], 3), "| raw:", round(raw.headline["f1_score"], 3))
check("G14 f1_score is case/punctuation-insensitive", norm.headline["f1_score"] >= raw.headline["f1_score"] and norm.headline["f1_score"] > 0.6)

## 3 · Tiny Aya: native tools are refused with advice, prompt mode works

Tiny Aya's chat template has no tool slot. Native mode would silently drop the schemas, so AuditKIT **refuses** it with a `CapabilityError` that says what to do. `ToolCallAdapter(mode="prompt")` writes the schemas into the system message instead, and that works with any text model.

In [ ]:
if HF_TOKEN:
    aya = HFGenModel(TINY_AYA, device=DEVICE, name=f"hf:{TINY_AYA}", token=HF_TOKEN)
    try:
        ak.evaluate(TASKS[:1], model=aya, adapter=ak.ToolCallAdapter(), scorers=[ak.ToolCallF1()])
        refused = False
    except CapabilityError as e:
        refused = True
        print("refused, as expected:", str(e)[:240])
    r = ak.evaluate(TASKS, model=aya, adapter=ak.ToolCallAdapter(mode="prompt"), scorers=SCORERS(), config=RunConfig(max_tokens=256, temperature=0.0))
    print("prompt mode:", {k: round(v, 2) for k, v in sorted(r.headline.items())})
    check("Tiny Aya: native refused with advice", refused)
    check("Tiny Aya: prompt-mode tool calls scored", "tool_call_f1" in r.headline)
    del aya; torch.cuda.empty_cache() if torch.cuda.is_available() else None
else:
    check("Tiny Aya: native refused with advice", False, skip="HF_TOKEN")
    check("Tiny Aya: prompt-mode tool calls scored", False, skip="HF_TOKEN")

## 4 · OpenAI-style string arguments in Command R7B's template (G11)

A conversation in OpenAI format carries each call's `function.arguments` as a **JSON string**, as AgentTune transcripts and every OpenAI client do. Command R7B's template writes `parameters` with `tojson`, so a string came out double-encoded (`"{\"a\": 2}"`), and the model saw malformed history.

`hf:` and `vllm:` now decode the strings first (`auditkit.model.template_messages`). Below, the **real Command R7B template** (shipped in the repo's test fixtures, so no gated download is needed) renders the same history both ways.

In [ ]:
from transformers import AutoTokenizer
from auditkit.model import template_messages
tok = AutoTokenizer.from_pretrained(QWEN)
tok.chat_template = (REPO / "tests/fixtures/chat_templates/command_r7b_tool_use.jinja").read_text()
history = [{"role": "user", "content": "What is 2+3?"},
           {"role": "assistant", "content": "", "tool_calls": [{"id": "call_1", "type": "function",
             "function": {"name": "add", "arguments": '{"a": 2, "b": 3}'}}]},
           {"role": "tool", "tool_call_id": "call_1", "content": "5"}]
add_tool = [{"type": "function", "function": {"name": "add", "parameters": {"type": "object"}}}]
before = tok.apply_chat_template(history, tools=add_tool, tokenize=False, add_generation_prompt=True)
after = tok.apply_chat_template(template_messages(history), tools=add_tool, tokenize=False, add_generation_prompt=True)
snip = lambda s: s[s.rfind('"parameters":'):s.rfind('"parameters":') + 40]      # the rendered call, not the template's instructions
print("raw history      :", snip(before))
print("template_messages:", snip(after))
check("G11 string arguments render as an object", '"parameters": {"a": 2, "b": 3}' in after and '\\"a\\"' not in after)

## 5 · Cohere's tool-call formats (G7–G10)

Cohere models write tool calls in their own formats:
- Command R7B: `<|START_ACTION|>[…]<|END_ACTION|>`;
- Command-R: `Action: ```json […]````.

Each call is `{"tool_call_id", "tool_name", "parameters"}`. These are Cohere's documented output formats, parsed with `ak.parse_tool_calls` and imported with the EventLog importer.

In [ ]:
from auditkit.agent_eval import episode_from_eventlog
calls = lambda text: [(c.name, c.arguments, c.parse_error) for c in ak.parse_tool_calls(text)]
R7B_REPLY = '<|START_ACTION|>[{"tool_call_id": "0", "tool_name": "get_weather", "parameters": {"city": "Paris"}}, {"tool_call_id": "1", "tool_name": "get_weather", "parameters": {"city": "Rome"}}]<|END_ACTION|>'
print("R7B parallel call   :", calls(R7B_REPLY))
direct = 'Action: ```json\n[{"tool_name": "directly-answer", "parameters": {}}]\n```'
print("directly-answer     :", calls(direct), " <- G9: a no-tool answer, not a call")
cut = '<|START_ACTION|>[{"tool_name": "get_time", "parameters": {}}, {"tool_name": "get_weather", "parameters": {"ci'
print("truncated list      :", calls(cut), " <- G10: the cut call is reported, not dropped")
ep = episode_from_eventlog({"id": "e", "tier": "full", "events": [
    {"kind": "tool_call", "payload": {"action": {"tool_calls": [{"tool_call_id": "7", "tool_name": "get_time"}]}}},   # G8: no parameters
    {"kind": "tool_result", "payload": {"tool_call_id": "7", "output": "14:05"}},
    {"kind": "text", "payload": {"text": "It's 14:05."}}]})
print("parameterless call  :", ep.to_trace()["tool_calls"], " <- G8")
print("call/result pairing :", ep.coverage["call_result_pairing"], " <- G7: Cohere's tool_call_id is kept")
check("G9 directly-answer is not a call", calls(direct) == [])
check("G10 truncated action list reported", len(calls(cut)) == 2 and calls(cut)[1][2] == "truncated")
check("G8 parameterless Cohere call counted", ep.to_trace()["tool_calls"] == [[{"tool_call_id": "7", "tool_name": "get_time"}]])
check("G7 Cohere call ids pair with results", ep.coverage["call_result_pairing"] == "observed")

## 6 · Command R7B, live native tool calling

Command R7B's template renders tools, and it answers in its action format. The markers are special tokens and are stripped on decode; AuditKIT reads the action list either way. It needs a 24 GB+ GPU.

In [ ]:
if HF_TOKEN and GPU_GB >= 23:
    r7b = HFGenModel(COMMAND_R7B, device=DEVICE, name=f"hf:{COMMAND_R7B}", token=HF_TOKEN)
    r = ak.evaluate(TASKS, model=r7b, adapter=ak.ToolCallAdapter(), scorers=SCORERS(), config=RunConfig(max_tokens=384, temperature=0.0))
    print({k: round(v, 2) for k, v in sorted(r.headline.items())})
    for p in r.predictions:
        print(f"  {p.sample_id:10s} -> {(p.raw_output or '')[:110]!r}")
    check("Command R7B native tool calls scored", r.headline.get("tool_call_f1", 0) > 0)
    del r7b; torch.cuda.empty_cache()
else:
    check("Command R7B native tool calls scored", False, skip=f"needs HF_TOKEN and 24 GB+ GPU (this: {GPU_GB:.0f} GB)")

## 7 · Cohere's hosted API through `api:`

Cohere serves an OpenAI-compatible endpoint, so `api:` sends the tools natively and reads **structured** `tool_calls`; nothing is parsed from text. The multi-turn sample puts a tool result in the history, so the right move is to answer.

In [ ]:
if os.environ.get("CO_API_KEY"):
    from auditkit.model.api_gen import APIModel
    co = APIModel(COHERE_API_MODEL, api_base=COHERE_BASE, api_key=os.environ["CO_API_KEY"], name=f"api:cohere/{COHERE_API_MODEL}")
    followup = Sample(id="answer_from_result", input="What's the weather in Paris?", tools=TOOLS, expected_tool_calls=[],
                      metadata={"messages": history[:0] + [
                          {"role": "user", "content": "What's the weather in Paris?"},
                          {"role": "assistant", "content": "", "tool_calls": [{"id": "c1", "type": "function",
                           "function": {"name": "get_weather", "arguments": '{"city": "Paris"}'}}]},
                          {"role": "tool", "tool_call_id": "c1", "content": '{"temp_c": 18, "sky": "sunny"}'}]})
    r = ak.evaluate(TASKS + [followup], model=co, adapter=ak.ToolCallAdapter(), scorers=SCORERS(),
                    config=RunConfig(max_tokens=256, temperature=0.0, concurrency=1, max_retries=6, retry_delay=10))
    print({k: round(v, 2) for k, v in sorted(r.headline.items())}, "| errors:", r.errors[:2])
    check("Cohere API native tool calls scored", r.headline.get("tool_call_f1", 0) > 0 and not r.errors)
else:
    check("Cohere API native tool calls scored", False, skip="CO_API_KEY")

## 8 · The harness drives a *text* model (G1, G2, #27)

In harness mode AuditKIT owns the loop: it sends the task, executes each tool call against a test double, feeds the result back, and checks the **final state** with an oracle.
- **G1:** an `hf:` model's calls arrive as text, and they're now parsed and executed. They used to be mistaken for the final answer.
- **G2:** `AgentEvalSpec(config=RunConfig(max_tokens=...))` reaches every harness request.
- **#27:** the history fed back holds each call once. The text markup is removed from `content`.

In [ ]:
from auditkit.agent_eval import AgentCase, AgentEvalRunner, AgentEvalSpec, FinalStateAssertion
BANK_TOOLS = [
    {"type": "function", "function": {"name": "transfer", "description": "Move money between the user's accounts",
     "parameters": {"type": "object", "properties": {"src": {"type": "string"}, "dst": {"type": "string"}, "amount": {"type": "number"}},
                    "required": ["src", "dst", "amount"]}}},
    {"type": "function", "function": {"name": "get_balance", "description": "Balance of one account",
     "parameters": {"type": "object", "properties": {"account": {"type": "string"}}, "required": ["account"]}}},
]
class Bank:
    def __init__(self):
        self.acc = {"checking": 1000.0, "savings": 0.0}
    def __call__(self, name, args):
        if name == "get_balance":
            return json.dumps({"balance": self.acc.get(args.get("account"))})
        if name == "transfer":
            a = float(args["amount"]); self.acc[args["src"]] -= a; self.acc[args["dst"]] += a
            return json.dumps({"ok": True, "balances": self.acc})
        return "unknown tool"
    def snapshot(self):
        return dict(self.acc)
case = AgentCase(id="move-250", task="Move 250 from checking to savings, then tell me the new savings balance.",
                 allowed_tools=BANK_TOOLS, outcome=FinalStateAssertion("savings", equals=250.0))
# no wrapper around the hf: model; the budget comes from config (the harness doesn't forward chat_template_kwargs, so Qwen3 thinks here)
res = AgentEvalRunner().run(AgentEvalSpec(cases=[case], mode="harness", agent=qwen, agent_opts={"tool_env": Bank(), "max_steps": 5},
                                          scorers=["tool_call_validity", "agent_loop_detection"], config=RunConfig(max_tokens=1536, temperature=0.0)))
row, ep = res.rows[0], res.episodes[0]
print("outcome:", row.outcome["verdict"], "| stop:", row.stop_reason, "| final state:", ep.final_state)
print("calls executed:", [(e.payload.get("function") or e.payload).get("name") for e in ep.tool_call_events()])
check("G1/G2 harness executes an hf: model's text tool calls", row.outcome["verdict"] == "success")

## 9 · AgentTune TraceLogger files: calls without names are unscored, not 0.0 (G3–G5)

`train_grpo`'s TraceLogger records each call's query and result, but **not the tool name**. Name-based metrics can't be computed from that, so AuditKIT:
- marks the calls `tool_calls_unavailable` (it used to say zero calls, which scored a measured 0.0);
- `inspect_agenttune` stops offering those metrics;
- `rescore` says why they were left out.

In [ ]:
import tempfile
from auditkit.agent_eval import AgentCase, inspect_agenttune, rescore
rec = {"question": "When was the bank founded?", "final_answer": "<answer>1887</answer>", "gold_answer": "1887", "reward": 0.95,
       "n_tool_calls": 2, "retrieved_chunk_ids": ["doc-2::3"], "gold_chunk_ids": ["doc-2::3"],
       "tool_calls": [{"query": {"query": "bank founded"}, "result": "{'search_corpus': '[chunk_id=doc-2::3] Founded in 1887.'}"},
                      {"query": {"query": "founding year"}, "result": "{'search_corpus': '[chunk_id=doc-2::3] Founded in 1887.'}"}]}
path = Path(tempfile.mkdtemp()) / "trace.jsonl"; path.write_text(json.dumps(rec) + "\n")
ep = ak.episodes_from_agenttune(str(path))[0]
trace = ep.to_trace()
print("recorded calls:", ep.counters.get("n_tool_calls"), "| trace:", {k: v for k, v in trace.items() if k.startswith("tool_calls")})
eligible = inspect_agenttune(str(path))["eligible_metrics"]
print("eligible metrics:", eligible)
ep.case_id = "t0"
row = rescore([ep], ["tool_call_f1"], cases=[AgentCase(id="t0", task="q", reference_turns=[{"name": "search_corpus", "arguments": {}}])]).rows[0]
print("rescore:", {"ineligible": row.ineligible})
check("G3 nameless calls are unavailable, not zero", trace.get("tool_calls_unavailable") is True and "tool_calls" not in trace)
check("G4 inspect doesn't offer name-based metrics", not {"tool_call_f1", "trajectory_match"} & set(eligible))
check("G5 rescore says why", row.ineligible.get("tool_call_f1") == "trace.tool_call_names")

## 10 · An `agent:` reply without a transcript is unscored (G6)

A deployed agent that returns only `{"output": ...}` has hidden what it did with tools, so tool metrics can't be computed. That is now **unscored**, whether or not the request offered tools. A server-side tool agent is never offered tools, and it used to get a measured "called nothing". An explicit `"tool_calls": []` is still evidence of no calls.

In [ ]:
REPLY = {"body": {"output": "4"}}
class AnswerOnly(BaseHTTPRequestHandler):
    def do_POST(self):
        self.rfile.read(int(self.headers["Content-Length"]))
        data = json.dumps(REPLY["body"]).encode()
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data))); self.end_headers(); self.wfile.write(data)
    def log_message(self, *a):
        pass
srv = ThreadingHTTPServer(("127.0.0.1", 0), AnswerOnly); threading.Thread(target=srv.serve_forever, daemon=True).start()
BASE = f"agent:http://127.0.0.1:{srv.server_address[1]}"
s = Sample(id="calc", input="What is 2+2?", expected_tool_calls=[])
hidden = ak.evaluate([s], model=BASE + "/answer-only", scorers=[ak.ToolCallF1()])
REPLY["body"] = {"output": "4", "tool_calls": []}
# a different URL is a different model spec, so the prediction cache doesn't hand back the first reply
explicit = ak.evaluate([s], model=BASE + "/explicit", scorers=[ak.ToolCallF1()])
print("answer only        :", hidden.headline or "tool_call_f1 not scored (tool use unobservable)")
print("explicit no calls  :", explicit.headline)
check("G6 answer-only agent reply is unscored", "tool_call_f1" not in hidden.headline and explicit.headline.get("tool_call_f1") == 1.0)

## 11 · Setup errors are raised at once (G16)

A missing model, a failed import or a wrong argument can't be fixed by retrying. They used to be retried 3 times and surfaced as `generate failed after 3 retries`, with the real cause buried in a chained traceback. Now the original error is raised immediately.

In [ ]:
t0 = time.time()
try:
    ak.evaluate([Sample(input="hi", target="x")], model=HFGenModel("auditkit-examples/this-model-does-not-exist", device="cpu"), scorers=["exact_match"])
    err = None
except Exception as e:  # noqa: BLE001
    err = e
elapsed = time.time() - t0
print(f"{type(err).__name__} after {elapsed:.1f}s:", str(err)[:200])
check("G16 setup error raised at once, not retried", err is not None and "retries" not in str(err) and elapsed < 60)

## 12 · Checklist

In [ ]:
display(pd.Series(CHECKS, name="result").to_frame())
print(sum(v == "PASS" for v in CHECKS.values()), "passed |", sum(v == "FAIL" for v in CHECKS.values()), "failed |",
      sum(v.startswith("SKIP") for v in CHECKS.values()), "skipped")

**Covered by unit tests rather than a live demo:**
- **G12:** one BOS token per prompt in a batch mixing raw and chat-templated prompts, in `tests/test_cohere_hf_fix_plan.py`.
- **G15, G17–G19:** `check_compat` rules for accelerate, SGLang on Python 3.13, and the torchaudio mismatch. The `check_compat()` output at the top shows whichever apply to this runtime.